In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

TARGET = 'Subscribed'
ID_COL = 'id'
SEEDS = [42, 2026]
BASE_CAT = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
MONTH_ORDER = {
    'jan': 1,
    'feb': 2,
    'mar': 3,
    'apr': 4,
    'may': 5,
    'jun': 6,
    'jul': 7,
    'aug': 8,
    'sep': 9,
    'oct': 10,
    'nov': 11,
    'dec': 12,
}

CATBOOST_PARAMS = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'iterations': 2500,
    'learning_rate': 0.04,
    'depth': 6,
    'l2_leaf_reg': 6.0,
    'random_strength': 1.0,
    'bagging_temperature': 0.2,
    'bootstrap_type': 'Bayesian',
    'od_type': 'Iter',
    'od_wait': 200,
    'allow_writing_files': False,
    'verbose': False,
}


def locate_file(filename):
    candidates = [
        Path('/kaggle/input/fiicode-2026-ai-competition') / filename,
        Path('/kaggle/input/competitions/fiicode-2026-ai-competition') / filename,
        Path(filename),
        Path.cwd() / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Could not find {filename}. Checked: {candidates}')


def output_path(filename):
    kaggle_working = Path('/kaggle/working')
    if kaggle_working.exists():
        return kaggle_working / filename
    return Path.cwd() / filename


def rank_norm(values):
    return pd.Series(values).rank(method='average', pct=True).to_numpy()


def raw_features(df):
    df = df.copy().drop(columns=[ID_COL])
    for col in BASE_CAT:
        df[col] = df[col].astype(str)
    return df, BASE_CAT.copy()


def minimal_features(df):
    df = df.copy().drop(columns=[ID_COL])
    for col in BASE_CAT:
        df[col] = df[col].astype(str)

    df['month_num'] = df['month'].map(MONTH_ORDER)
    df['has_previous'] = (df['pdays'] != -1).astype(int)
    df['pdays_clean'] = np.where(df['pdays'] == -1, 999, df['pdays'])
    df['balance_log'] = np.sign(df['balance']) * np.log1p(np.abs(df['balance']))
    df['duration_log'] = np.log1p(df['duration'])
    df['campaign_log'] = np.log1p(df['campaign'])
    df['previous_log'] = np.log1p(df['previous'])
    df['job_education'] = (df['job'] + '__' + df['education']).astype(str)
    df['contact_month'] = (df['contact'] + '__' + df['month']).astype(str)
    df['age_band'] = pd.cut(
        df['age'],
        bins=[17, 25, 35, 45, 55, 65, np.inf],
        labels=['18-25', '26-35', '36-45', '46-55', '56-65', '66+'],
        include_lowest=True,
    ).astype(str)

    cat_cols = BASE_CAT + ['job_education', 'contact_month', 'age_band']
    for col in cat_cols:
        df[col] = df[col].astype(str)
    return df, cat_cols


def curated_features(df):
    df = df.copy().drop(columns=[ID_COL])
    for col in BASE_CAT:
        df[col] = df[col].astype(str)

    df['month_num'] = df['month'].map(MONTH_ORDER)
    df['has_previous'] = (df['pdays'] != -1).astype(int)
    df['pdays_clean'] = np.where(df['pdays'] == -1, 999, df['pdays'])
    df['balance_log'] = np.sign(df['balance']) * np.log1p(np.abs(df['balance']))
    df['duration_log'] = np.log1p(df['duration'])
    df['campaign_log'] = np.log1p(df['campaign'])
    df['previous_log'] = np.log1p(df['previous'])
    df['job_education'] = (df['job'] + '__' + df['education']).astype(str)
    df['contact_month'] = (df['contact'] + '__' + df['month']).astype(str)
    df['contact_poutcome'] = (df['contact'] + '__' + df['poutcome']).astype(str)
    df['loan_profile'] = (df['default'] + '__' + df['housing'] + '__' + df['loan']).astype(str)
    df['age_band'] = pd.cut(
        df['age'],
        bins=[17, 25, 35, 45, 55, 65, np.inf],
        labels=['18-25', '26-35', '36-45', '46-55', '56-65', '66+'],
        include_lowest=True,
    ).astype(str)
    df['duration_band'] = pd.cut(
        df['duration'],
        bins=[0, 60, 120, 240, 480, np.inf],
        labels=['0-60', '61-120', '121-240', '241-480', '480+'],
        include_lowest=True,
    ).astype(str)
    df['balance_band'] = pd.cut(
        df['balance'],
        bins=[-np.inf, 0, 500, 1500, 5000, np.inf],
        labels=['neg', '0-500', '500-1500', '1500-5000', '5000+'],
        include_lowest=True,
    ).astype(str)
    df['pdays_band'] = np.select(
        [df['pdays'] == -1, df['pdays'] <= 7, df['pdays'] <= 30, df['pdays'] <= 90],
        ['never', '1w', '1m', '3m'],
        default='90+',
    )

    cat_cols = BASE_CAT + [
        'job_education',
        'contact_month',
        'contact_poutcome',
        'loan_profile',
        'age_band',
        'duration_band',
        'balance_band',
        'pdays_band',
    ]
    for col in cat_cols:
        df[col] = df[col].astype(str)
    return df, cat_cols


def fit_seed_ensemble(x_train, x_test, y, cat_cols):
    model_oof = np.zeros(len(x_train), dtype=float)
    model_test = np.zeros(len(x_test), dtype=float)

    for seed in SEEDS:
        seed_oof = np.zeros(len(x_train), dtype=float)
        seed_test = np.zeros(len(x_test), dtype=float)
        folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

        for train_idx, valid_idx in folds.split(x_train, y):
            model = CatBoostClassifier(**CATBOOST_PARAMS, random_seed=seed)
            model.fit(
                x_train.iloc[train_idx],
                y.iloc[train_idx],
                eval_set=(x_train.iloc[valid_idx], y.iloc[valid_idx]),
                cat_features=cat_cols,
                use_best_model=True,
                verbose=False,
            )
            seed_oof[valid_idx] = model.predict_proba(x_train.iloc[valid_idx])[:, 1]
            seed_test += model.predict_proba(x_test)[:, 1] / folds.n_splits

        seed_score = roc_auc_score(y, seed_oof)
        print(f'Seed {seed} AUC: {seed_score:.6f}')
        model_oof += seed_oof / len(SEEDS)
        model_test += seed_test / len(SEEDS)

    return model_oof, model_test


train_path = locate_file('train.csv')
test_path = locate_file('test.csv')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
y = train[TARGET].astype(int)

feature_builders = [
    ('raw', raw_features),
    ('minimal', minimal_features),
    ('curated', curated_features),
]

results = {}
for model_name, builder in feature_builders:
    x_train, cat_cols = builder(train.drop(columns=[TARGET]))
    x_test, _ = builder(test)
    model_oof, model_test = fit_seed_ensemble(x_train, x_test, y, cat_cols)
    model_score = roc_auc_score(y, model_oof)
    results[model_name] = {'oof': model_oof, 'test': model_test, 'score': model_score}
    print(f'{model_name} model AUC: {model_score:.6f}')

best_score = -1.0
best_weights = None
for w_raw in np.arange(0.10, 0.90, 0.05):
    for w_min in np.arange(0.10, 0.90, 0.05):
        w_cur = 1.0 - w_raw - w_min
        if w_cur < 0.05:
            continue
        blend_oof = (
            w_raw * rank_norm(results['raw']['oof'])
            + w_min * rank_norm(results['minimal']['oof'])
            + w_cur * rank_norm(results['curated']['oof'])
        )
        blend_score = roc_auc_score(y, blend_oof)
        if blend_score > best_score:
            best_score = blend_score
            best_weights = (float(w_raw), float(w_min), float(w_cur))

w_raw, w_min, w_cur = best_weights
print(f'Best blend weights: raw={w_raw:.2f}, minimal={w_min:.2f}, curated={w_cur:.2f}')
print(f'Best blend AUC: {best_score:.6f}')

blend_test = (
    w_raw * rank_norm(results['raw']['test'])
    + w_min * rank_norm(results['minimal']['test'])
    + w_cur * rank_norm(results['curated']['test'])
)

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: np.clip(blend_test, 1e-6, 1.0 - 1e-6),
})
save_path = output_path('submission.csv')
submission.to_csv(save_path, index=False)
print(f'Submission saved to: {save_path}')
submission.head()
